# Conhecendo os dados

**Pergunta do projeto:** Entre as reclamações NIP de 2025, quais assuntos mais geram
reclamação, e esse ranking muda entre os tipos de operadora?

**Fonte:** Dados abertos da ANS, reclamações de beneficiários (NIP), ano de 2025.
Arquivo atualizado pela ANS em 05/10/2026 e baixado em 11/10/2026.

Neste notebook eu abro o arquivo pela primeira vez e confiro se o conteúdo
bate com o dicionário de dados, antes de fazer qualquer alteração.

In [1]:
import pandas as pd

caminho = '../dados/pda-014-demandas_dos_consumidores_reclamacao_beneficiarios-2025.csv'

df = pd.read_csv(caminho, sep=';')

print(f'Linhas: {df.shape[0]}')
print(f'Colunas: {df.shape[1]}')

Linhas: 336589
Colunas: 21


O arquivo tem **336.589 reclamações** e **21 colunas**. O número de colunas
bate com o dicionário de dados da ANS.

In [2]:
df.head()

,NR_DEMANDA,DT_ATENDIMENTO,SITUACAO_DEMANDA,ENCAMINHADO_OUVIDORIA,MEIO_ATENDIMENTO,ASSUNTO,REGISTRO_OPERADORA,RAZÃO_SOCIAL_OPERADORA,MODALIDADE_OPERADORA,TIPO_PLANO_CONTRATADO,...,COBERTURAS_CONTRATADAS,DT_NASCIMENTO_BENEFICIARIO,SEXO_BENEFICIARIO,UF_BENEFICIARIO,CD_MUNICIPIO_BENEFICIARIO,PROCEDIMENTO,UF_LOCAL_ATENDIMENTO,MUNICIPIO_LOCAL_ATENDIMENTO,CD_MUNICIPIO_LOCAL_ATENDIMENTO,NR_PROCESSO
0,13387932,2025-03-31,Prazo de Resposta do Consumidor expirado,NÃO,Site,Produto ou Plano >> Cobertura >> Reembolso,6246.0,SUL AMERICA COMPANHIA DE SEGURO SAÚDE,Seguradora Especializada em Saúde,Coletivo por Adesão,...,Ambulatorial + Hospitalar c/ Obstetrícia (A) +...,1969-06-25,F,SP,354990.0,NaN,SP,SAO JOSE DOS CAMPOS ...,354990.0,NaN
1,13448233,2025-05-14,Alteração Manual,NÃO,Telefone,Produto ou Plano >> Cobertura >> Prazos Máximo...,368253.0,HAPVIDA ASSISTENCIA MEDICA S.A.,Medicina de Grupo,NaN,...,NaN,2007-03-08,M,CE,230440.0,NaN,CE,FORTALEZA ...,230440.0,NaN
2,13683597,2025-10-31,Prazo de Resposta do Consumidor expirado,NÃO,Site,Produto ou Plano >> Cobertura >> Reembolso,5711.0,BRADESCO SAÚDE S.A.,Seguradora Especializada em Saúde,Individual ou Familiar,...,Ambulatorial + Hospitalar c/ Obstetrícia (A) +...,1975-06-04,F,SP,352440.0,NaN,SP,SAO PAULO ...,355030.0,NaN
3,13590688,2025-08-28,Encaminhado à GEFIN para Cobrança,NÃO,Telefone,Produto ou Plano >> Cobertura >> Rol de Proced...,343463.0,PLAMED PLANO DE ASSISTENCIA MEDICA LTDA,Medicina de Grupo,NaN,...,NaN,1964-11-23,M,SE,280030.0,NaN,SE,ARACAJU ...,280030.0,33910.011741/2026-50
4,13495979,2025-06-17,Prazo de Resposta do Consumidor expirado,NÃO,Telefone,Produto ou Plano >> Mensalidades e Reajustes >...,417173.0,QUALICORP ADMINISTRADORA DE BENEFÍCIOS S.A.,Administradora de Benefícios,Coletivo por Adesão,...,Ambulatorial + Hospitalar c/ Obstetrícia (A) +...,2016-04-26,M,MG,316990.0,NaN,NaN,NaN,NaN,NaN


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 336589 entries, 0 to 336588
Data columns (total 21 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   NR_DEMANDA                      336589 non-null  int64  
 1   DT_ATENDIMENTO                  336589 non-null  str    
 2   SITUACAO_DEMANDA                336589 non-null  str    
 3   ENCAMINHADO_OUVIDORIA           336589 non-null  str    
 4   MEIO_ATENDIMENTO                336589 non-null  str    
 5   ASSUNTO                         336587 non-null  str    
 6   REGISTRO_OPERADORA              336531 non-null  float64
 7   RAZÃO_SOCIAL_OPERADORA          336531 non-null  str    
 8   MODALIDADE_OPERADORA            336531 non-null  str    
 9   TIPO_PLANO_CONTRATADO           331386 non-null  str    
 10  EPOCA_CONTRATACAO_PLANO         336589 non-null  str    
 11  COBERTURAS_CONTRATADAS          331386 non-null  str    
 12  DT_NASCIMENTO_BENEFICIARIO 

## O que o info() mostrou

- `DT_ATENDIMENTO` foi lida como texto (`str`), e não como data.
  Vou converter na etapa de limpeza.
- `NR_DEMANDA` foi lido como número inteiro (`int64`).
- `REGISTRO_OPERADORA` foi lido como número decimal (`float64`), mas é um código.
  Também vou ajustar na limpeza.
- `PROCEDIMENTO` está preenchido em apenas 3.495 das 336.589 reclamações
  (cerca de 1%).

**Observação:** o dicionário da ANS escreve dois nomes de campo diferente do
arquivo: `RAZÃO SOCIAL_OPERADORA` (no arquivo, `RAZÃO_SOCIAL_OPERADORA`) e
`TIPOS_COBERTURAS_CONTRATADAS` (no arquivo, `COBERTURAS_CONTRATADAS`). Por isso,
o projeto usa os nomes que aparecem no arquivo.

## Decisão: ajuste da pergunta

A pergunta original era sobre procedimentos. Como o procedimento só aparece
em 1% das reclamações, e esse 1% pode não representar o total, troquei o foco
para o `ASSUNTO`, que está preenchido em praticamente 100% dos registros.

**Nova pergunta:** Entre as reclamações NIP de 2025, quais assuntos mais geram
reclamação, e esse ranking muda entre os tipos de operadora?

In [4]:
pd.set_option('display.max_colwidth', None)

print(f"Assuntos diferentes: {df['ASSUNTO'].nunique()}")

df['ASSUNTO'].value_counts().head(10)

Assuntos diferentes: 65


ASSUNTO
Produto ou Plano >> Cobertura >> Regras para Acesso aos Atendimentos                          79983
Produto ou Plano >> Cobertura >> Prazos Máximos para Atendimento                              49758
Produto ou Plano >> Cobertura >> Reembolso                                                    45513
Produto ou Plano >> Cobertura >> Rol de Procedimentos e Cobertura Contratual                  43209
Produto ou Plano >> Cobertura >> Rede de Atendimento (rede conveniada)                        39133
Produto ou Plano >> Contratos e Regulamentos >> Suspensão e Rescisão Contratuais              19132
Produto ou Plano >> Mensalidades e Reajustes >> Mensalidade ou Outras Cobranças               13425
Produto ou Plano >> Contratos e Regulamentos >> Contratação/Adesão e Vigência Contratual       5509
Produto ou Plano >> Cobertura >> Carência                                                      5505
Produto ou Plano >> Contratos e Regulamentos >> Itens Obrigatórios e Cláusulas Contratuais  

## Conhecendo a coluna ASSUNTO

Existem **65 assuntos diferentes**. Cada assunto tem três níveis separados
por `>>`, do mais geral para o mais específico. Exemplo:
`Produto ou Plano >> Cobertura >> Prazos Máximos para Atendimento`.

Os cinco assuntos mais frequentes são todos do grupo **Cobertura**.

In [5]:
niveis = df['ASSUNTO'].str.split(' >> ', expand=True)

print(f"Formato do resultado: {niveis.shape}")
niveis.head()

Formato do resultado: (336589, 3)


,0,1,2
0,Produto ou Plano,Cobertura,Reembolso
1,Produto ou Plano,Cobertura,Prazos Máximos para Atendimento
2,Produto ou Plano,Cobertura,Reembolso
3,Produto ou Plano,Cobertura,Rol de Procedimentos e Cobertura Contratual
4,Produto ou Plano,Mensalidades e Reajustes,Reajuste por Variação de Custos


In [6]:
df['ASSUNTO_NIVEL1'] = niveis[0]
df['ASSUNTO_NIVEL2'] = niveis[1]
df['ASSUNTO_NIVEL3'] = niveis[2]

print(f"Colunas agora: {df.shape[1]}")

Colunas agora: 24


In [7]:
df['ASSUNTO_NIVEL1'].value_counts(dropna=False)

ASSUNTO_NIVEL1
Produto ou Plano                   333478
REDE DE ATENDIMENTO - DIPRO          2161
Operadora                             826
ANS - Agência Nacional de Saúde       122
NaN                                     2
Name: count, dtype: int64

In [8]:
(df['ASSUNTO_NIVEL2'].value_counts(normalize=True) * 100).round(1)

ASSUNTO_NIVEL2
Cobertura                                                               79.9
Contratos e Regulamentos                                                13.3
Mensalidades e Reajustes                                                 5.8
Rede Não Hospitalar                                                      0.3
Rede Hospitalar                                                          0.3
Regras para Atendimento                                                  0.2
Informações                                                              0.0
Ouvidoria da Operadora                                                   0.0
Regras Contratuais contrárias a Legislação                               0.0
Registro na ANS                                                          0.0
Regimes Especiais - Desequilíbrios Técnicos ou Econômico-Financeiros     0.0
Transferência de Carteira                                                0.0
Contratualização                                             

In [9]:
valores = df['ASSUNTO_NIVEL2'].dropna().unique()

for valor in valores:
    if 'Hospitalar' in valor:
        print(repr(valor), len(valor))

'Rede Não Hospitalar' 19
'Rede Hospitalar' 15
'Rede Não Hospitalar ' 20


## Níveis do assunto

Separei o `ASSUNTO` em três colunas: `ASSUNTO_NIVEL1`, `ASSUNTO_NIVEL2` e
`ASSUNTO_NIVEL3`. Nenhuma linha foi perdida (336.589).

- **Nível 1:** 99% das reclamações são de "Produto ou Plano".
- **Nível 2:** Cobertura concentra **79,9%** das reclamações. Junto com
  Contratos e Regulamentos (13,3%) e Mensalidades e Reajustes (5,8%),
  os três grupos somam 99%.

**Problema encontrado:** a categoria "Rede Não Hospitalar" aparece duas vezes,
porque uma das versões tem um espaço no final. Vou corrigir na limpeza,
removendo espaços sobrando em todas as colunas de texto.

In [10]:
df['MODALIDADE_OPERADORA'].value_counts(dropna=False)

MODALIDADE_OPERADORA
Medicina de Grupo                    142881
Cooperativa Médica                    97634
Seguradora Especializada em Saúde     65493
Autogestão                            20367
Administradora de Benefícios           4298
Filantropia                            4029
Odontologia de Grupo                   1602
Cooperativa odontológica                223
NaN                                      58
Seguradora                                4
Name: count, dtype: int64

In [11]:
df['SITUACAO_DEMANDA'].value_counts(dropna=False)

SITUACAO_DEMANDA
Prazo de Resposta do Consumidor expirado                                                       183082
Alteração Manual                                                                                41884
Tentativa/Contato Consumidor Registrado                                                         33472
Resposta do Interlocutor à solução do Conflito (NIP RN 343)                                     28392
Demanda Finalizada                                                                              13969
Prazo de Resposta da Operadora expirado                                                         12792
Documento Enviado para Operadora                                                                11499
Solicitação de reabertura da Demanda para Classificação.                                         3503
Demanda encaminhada para Tratamento Técnico                                                      2157
Encaminhado à GEFIN para Cobrança                                

## Modalidade e situação

**Modalidade da operadora:** 9 tipos, com 58 reclamações sem modalidade
(as mesmas 58 sem registro e nome da operadora, a confirmar na limpeza).
Medicina de Grupo, Cooperativa Médica e Seguradora Especializada em Saúde
concentram a maior parte. Atenção: mais reclamações não significa operadora
pior, porque modalidades maiores têm mais clientes. Isso será tratado na análise.

**Situação da demanda:** não indica se a reclamação foi resolvida, e sim a etapa
atual do processo na ANS, com dezenas de valores diferentes. Como a pergunta
do projeto não depende da situação, essa coluna fica fora do escopo.

## Pendências para a limpeza

1. Converter `DT_ATENDIMENTO` de texto para data.
2. Converter `REGISTRO_OPERADORA` de número decimal para código.
3. Remover espaços sobrando em todas as colunas de texto.
4. Investigar as 2 reclamações sem `ASSUNTO` e as 58 sem dados da operadora.
5. Remover colunas pessoais que não serão usadas (como data de nascimento).